# EnvForge Colab GPU Benchmark

EnvForge + ML-Agents `NavigationFinal` をGoogle Colabで10万stepだけ実行し、CPU/T4/L4/A100などの速度差を見るための作業台です。

このnotebookは、学習性能ではなく実行速度の比較を目的にします。判断基準は `docs/implementation/colab-gpu-benchmark-plan.md` を参照します。

前提として、Windows側UnityからLinux用にエクスポートしたheadless buildをGoogle Drive上のフォルダに置いてください。前回の `ColabSmokeTest.ipynb` と同じく、Colab上ではDriveから `/content/unity_build` へコピーして、`xvfb-run` と `-force-glcore` で起動します。

## 1. Runtime確認

CPU条件では `Runtime > Change runtime type > Hardware accelerator: None` を選びます。

GPU条件ではT4、L4、A100など、Colabで選択または割り当て可能なGPUを使います。Premium GPUが必要な場合は、この段階でColabの課金が必要になります。

In [ ]:
!date -u
!python --version
!uname -a
!nvidia-smi || true
!lscpu | sed -n '1,24p'
!free -h


## 2. Google Driveをmount

`BENCH_ROOT` 配下に結果、Unity build、ログを置きます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BENCH_ROOT = Path('/content/drive/MyDrive/EnvForge/colab-bench')
BENCH_ROOT.mkdir(parents=True, exist_ok=True)
print(BENCH_ROOT)


## 3. 設定値

`REPO_URL`、`UNITY_BUILD_DRIVE_DIR`、`UNITY_EXECUTABLE_NAME` は実際の配置に合わせて変更してください。

`RUN_ID` は実行条件ごとに変えます。例: `bench-cpu-100k`, `bench-t4-100k`, `bench-l4-100k`, `bench-a100-100k`。

In [ ]:
REPO_URL = 'https://github.com/sayakaakioka/EnvForge.git'
REPO_BRANCH = 'main'
RUN_ID = 'bench-cpu-100k'

REPO_DIR = Path('/content/EnvForge')
UNITY_BUILD_DRIVE_DIR = Path('/content/drive/MyDrive/colab/EnvForgeNavigationFinal')
UNITY_BUILD_DIR = Path('/content/unity_build')
UNITY_EXECUTABLE_NAME = 'EnvForgeNavigationFinal.x86_64'
UNITY_EXECUTABLE = UNITY_BUILD_DIR / UNITY_EXECUTABLE_NAME

print('REPO_URL =', REPO_URL)
print('RUN_ID =', RUN_ID)
print('UNITY_BUILD_DRIVE_DIR =', UNITY_BUILD_DRIVE_DIR)
print('UNITY_EXECUTABLE =', UNITY_EXECUTABLE)


## 4. リポジトリ取得

private repositoryの場合は、Colab側で認証済みURLまたは手動アップロードを使ってください。

In [ ]:
if REPO_DIR.exists():
    !git -C {REPO_DIR} fetch --all --prune
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} rev-parse --short HEAD
!ls -la {REPO_DIR}


## 5. Python 3.10.12と依存関係

ML-Agents Python `1.1.0` はPython `>=3.10.1,<=3.10.12` が必要です。ここでは `uv` でPython 3.10.12を用意します。

Unityのカメラ/RenderTexture付きheadless起動では、前回のSmokeTestと同じく `xvfb` も入れます。

In [ ]:
!apt-get update -qq
!apt-get install -y xvfb libgtk-3-0
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.environ['HOME'] + '/.local/bin:' + os.environ['PATH']
!uv --version
!cd {REPO_DIR} && uv python install 3.10.12
!cd {REPO_DIR} && uv sync --python 3.10.12
!cd {REPO_DIR} && uv run python --version
!cd {REPO_DIR} && uv run python -m mlagents.trainers.learn --help | head -40


## 6. ベンチ用configを生成

GitHub上に`configs/ml-agents/navigation-final-bench-100k.yaml`がない場合でも、Colab上で`navigation-final.yaml`から10万step用configを生成します。

In [ ]:
TRAIN_CONFIG = REPO_DIR / 'configs/ml-agents/navigation-final-bench-100k.yaml'
BASE_TRAIN_CONFIG = REPO_DIR / 'configs/ml-agents/navigation-final.yaml'

base_config_text = BASE_TRAIN_CONFIG.read_text(encoding='utf-8')
bench_config_text = base_config_text.replace('max_steps: 1500000', 'max_steps: 100000')
if bench_config_text == base_config_text:
    raise ValueError('Expected max_steps: 1500000 in navigation-final.yaml')

TRAIN_CONFIG.write_text(bench_config_text, encoding='utf-8')
print('Generated', TRAIN_CONFIG)
!grep -nE 'max_steps|summary_freq|threaded' {TRAIN_CONFIG}


## 6. Unity headless buildをコピー

前回の `ColabSmokeTest.ipynb` と同じ方式で、Google Drive上のLinux buildフォルダを `/content/unity_build` へコピーします。

`UNITY_BUILD_DRIVE_DIR` が存在しない場合は、Windows側UnityでLinux用にexportしたbuildフォルダをGoogle Driveの該当パスへ配置してください。

In [ ]:
assert UNITY_BUILD_DRIVE_DIR.exists(), f'Missing Unity build folder: {UNITY_BUILD_DRIVE_DIR}'
!rm -rf {UNITY_BUILD_DIR}
!cp -r {UNITY_BUILD_DRIVE_DIR} {UNITY_BUILD_DIR}
!find {UNITY_BUILD_DIR} -maxdepth 2 -type f | head -50
!chmod +x {UNITY_EXECUTABLE}
!ls -lh {UNITY_EXECUTABLE}


## 7. Unity headless起動コマンドを用意

CameraSensor/RenderTextureを使うため、Unityは`xvfb-run`上で起動します。

ML-Agents trainerを先に待ち受け状態にしてからUnityを起動します。Sceneには通常の推論確認用model assetが入っているため、Colab学習時は`--envforge-train`を付け、model assetがあってもtrainer接続を優先します。

In [ ]:
import subprocess, time

UNITY_LOG = BENCH_ROOT / f'{RUN_ID}-unity.log'
unity_cmd = [
    'xvfb-run', '-s', '-screen 0 1024x768x24',
    str(UNITY_EXECUTABLE),
    '-batchmode',
    '-force-glcore',
    '--envforge-train',
    '-logFile', str(UNITY_LOG),
]
print('Unity command:')
print(' '.join(unity_cmd))


## 8. 監視ログを開始

GPUなし条件では `nvidia-smi` が失敗しますが、そのまま進めます。

In [ ]:
MONITOR_LOG = BENCH_ROOT / f'{RUN_ID}-monitor.log'
monitor_cmd = f"""
while true; do
  date -u
  nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw --format=csv,noheader,nounits || true
  top -bn1 | head -20
  free -h
  sleep 30
done
"""
monitor_proc = subprocess.Popen(['bash', '-lc', monitor_cmd], stdout=open(MONITOR_LOG, 'w'), stderr=subprocess.STDOUT)
print('Monitor pid:', monitor_proc.pid)


## 9. 10万stepベンチを実行

完了までのwall-clock時間を記録します。`results` はGoogle Driveへコピーします。

In [ ]:
import time, shutil

TRAIN_LOG = BENCH_ROOT / f'{RUN_ID}-trainer.log'
start = time.time()
train_cmd = [
    'uv', 'run', 'python', '-m', 'envforge_mlagents.navigation_strict.train',
    str(TRAIN_CONFIG.relative_to(REPO_DIR)),
    '--run-id', RUN_ID,
    '--time-scale', '10',
    '--force',
]
with open(TRAIN_LOG, 'w') as log_file:
    train_proc = subprocess.Popen(train_cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT)
    print('Trainer pid:', train_proc.pid)
    time.sleep(10)
    unity_proc = subprocess.Popen(unity_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    print('Unity pid:', unity_proc.pid)
    train_return_code = train_proc.wait()
elapsed = time.time() - start
print('Trainer return code:', train_return_code)
print('Elapsed seconds:', elapsed)

elapsed_path = BENCH_ROOT / f'{RUN_ID}-elapsed.txt'
elapsed_path.write_text(f'{elapsed}\n', encoding='utf-8')

!tail -120 {TRAIN_LOG}


## 10. 結果を保存して終了

ベンチ終了後は、compute unit消費を避けるためランタイムを切断します。

In [ ]:
try:
    monitor_proc.terminate()
except Exception as exc:
    print('Monitor terminate skipped:', exc)

try:
    unity_proc.terminate()
except Exception as exc:
    print('Unity terminate skipped:', exc)

RESULTS_DIR = REPO_DIR / 'results' / RUN_ID
DEST_RESULTS_DIR = BENCH_ROOT / RUN_ID
if RESULTS_DIR.exists():
    if DEST_RESULTS_DIR.exists():
        shutil.rmtree(DEST_RESULTS_DIR)
    shutil.copytree(RESULTS_DIR, DEST_RESULTS_DIR)
    print('Copied results to', DEST_RESULTS_DIR)
else:
    print('No results dir found:', RESULTS_DIR)

!ls -lh {BENCH_ROOT}


## 11. 比較表

各条件が終わったら、以下を埋めます。

| run id | runtime | GPU | elapsed sec | steps/sec | 備考 |
| --- | --- | --- | ---: | ---: | --- |
| bench-cpu-100k | CPU | None |  |  |  |
| bench-t4-100k | GPU | T4 |  |  |  |
| bench-l4-100k | GPU | L4 |  |  |  |
| bench-a100-100k | GPU | A100 |  |  |  |